In [2]:
import cv2
import numpy as np
import os
import pandas as pd
import re


# =========================================================
# ROAD DEFECT DETECTOR
# =========================================================
class RoadDefectDetector:

    def __init__(self):
        self.clahe = cv2.createCLAHE(
            clipLimit=3.0,
            tileGridSize=(8, 8)
        )

    def get_auto_roi(self, frame):

        h, w = frame.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        points = np.array([[
            [int(w * 0.02), h],
            [int(w * 0.08), int(h * 0.18)],
            [int(w * 0.92), int(h * 0.18)],
            [int(w * 0.98), h]
        ]], dtype=np.int32)

        cv2.fillPoly(mask, points, 255)
        return mask

    def enhance(self, frame):

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        enhanced = self.clahe.apply(blur)
        return enhanced

    def process_frame(self, frame):

        roi_mask = self.get_auto_roi(frame)
        enhanced = self.enhance(frame)

        road_only = cv2.bitwise_and(
            enhanced,
            enhanced,
            mask=roi_mask
        )

        result = frame.copy()
        detected_types = set()

        binary = cv2.adaptiveThreshold(
            road_only,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            15,
            5
        )

        threshold_output = binary.copy()

        kernel = np.ones((3, 3), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        morphology_output = binary.copy()

        contours, _ = cv2.findContours(
            binary,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for cnt in contours:

            area = cv2.contourArea(cnt)
            if area < 100:
                continue

            x, y, w, h = cv2.boundingRect(cnt)
            aspect_ratio = float(w) / h

            if area > 1500:
                hull = cv2.convexHull(cnt)
                cv2.drawContours(result, [hull], -1, (255, 0, 0), 3)
                detected_types.add("Pothole")

            elif aspect_ratio > 4.0 and w > 80:
                cv2.drawContours(result, [cnt], -1, (0, 0, 255), 2)
                detected_types.add("Transverse Crack")

            elif (aspect_ratio < 0.25 and h > 80):
                cv2.drawContours(result, [cnt], -1, (0, 255, 255), 2)
                detected_types.add("Edge Crack")

            elif 500 < area < 4000:
                cv2.drawContours(result, [cnt], -1, (0, 255, 0), 1)
                detected_types.add("Ravelling")

        for i, text in enumerate(detected_types):
            cv2.putText(
                result,
                f"DETECTED: {text}",
                (20, 40 + (i * 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        return result, enhanced, threshold_output, morphology_output


# =========================================================
# VIDEO PROCESSING PIPELINE
# =========================================================

video_input = r"D:\image processing project\enhanced.mp4"
video_output = r"D:\image processing project\detected_output.mp4"

base_output = r"D:\segmentation\outputs"

enhanced_folder = os.path.join(base_output, "enhanced_frames")
threshold_folder = os.path.join(base_output, "threshold_masks")
morphology_folder = os.path.join(base_output, "morphological_refinement")
final_folder = os.path.join(base_output, "final_outputs")
comparison_folder = os.path.join(base_output, "comparisons")

os.makedirs(enhanced_folder, exist_ok=True)
os.makedirs(threshold_folder, exist_ok=True)
os.makedirs(morphology_folder, exist_ok=True)
os.makedirs(final_folder, exist_ok=True)
os.makedirs(comparison_folder, exist_ok=True)

detector = RoadDefectDetector()

cap = cv2.VideoCapture(video_input)

fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    video_output,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

print("Processing Video... Press 'q' to stop.")

frame_count = 0

while True:

    ret, frame = cap.read()

    if not ret:

        retry = 0
        while retry < 5:
            ret, frame = cap.read()
            if ret:
                break
            retry += 1

        if not ret:
            print("End of video or unreadable frame reached.")
            break

    processed, enhanced_frame, threshold_mask, morphology_mask = detector.process_frame(frame)

    cv2.imwrite(os.path.join(enhanced_folder, f"enhanced_{frame_count:05d}.png"), enhanced_frame)
    cv2.imwrite(os.path.join(threshold_folder, f"threshold_{frame_count:05d}.png"), threshold_mask)
    cv2.imwrite(os.path.join(morphology_folder, f"morphology_{frame_count:05d}.png"), morphology_mask)
    cv2.imwrite(os.path.join(final_folder, f"final_{frame_count:05d}.png"), processed)

    enhanced_bgr = cv2.cvtColor(enhanced_frame, cv2.COLOR_GRAY2BGR)
    threshold_bgr = cv2.cvtColor(threshold_mask, cv2.COLOR_GRAY2BGR)
    morphology_bgr = cv2.cvtColor(morphology_mask, cv2.COLOR_GRAY2BGR)

    comparison = np.hstack([
        frame,
        enhanced_bgr,
        threshold_bgr,
        morphology_bgr,
        processed
    ])

    cv2.imwrite(os.path.join(comparison_folder, f"comparison_{frame_count:05d}.png"), comparison)

    out.write(processed)

    cv2.imshow("Automated Road Defect Detection", processed)

    frame_count += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done!")


# =========================================================
# EVALUATION PIPELINE
# =========================================================

pred_folder = r"D:\segmentation\outputs\final_outputs"
gt_folder   = r"D:\segmentation\ground truth masks"
output_dir  = r"D:\segmentation\outputs"

frame_csv_path = os.path.join(output_dir, "frame_results.csv")
overall_csv_path = os.path.join(output_dir, "overall_results.csv")

pred_files = sorted(os.listdir(pred_folder))
gt_files = os.listdir(gt_folder)

results = []


def extract_number(filename):
    nums = re.findall(r'\d+', filename)
    return nums[0] if nums else None


def compute_metrics(gt, pred):

    gt = (gt > 127).astype(np.uint8)
    pred = (pred > 127).astype(np.uint8)

    TP = np.sum((gt == 1) & (pred == 1))
    FP = np.sum((gt == 0) & (pred == 1))
    FN = np.sum((gt == 1) & (pred == 0))
    TN = np.sum((gt == 0) & (pred == 0))

    eps = 1e-7

    acc = (TP + TN) / (TP + TN + FP + FN + eps)
    prec = TP / (TP + FP + eps)
    rec = TP / (TP + FN + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    iou = TP / (TP + FP + FN + eps)

    return acc, prec, rec, f1, iou


valid_frames = 0

for pred_file in pred_files:

    pred_num = extract_number(pred_file)
    if pred_num is None:
        continue

    gt_match = None
    for g in gt_files:
        if pred_num in g:
            gt_match = g
            break

    if gt_match is None:
        continue

    pred_path = os.path.join(pred_folder, pred_file)
    gt_path = os.path.join(gt_folder, gt_match)

    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

    if pred is None or gt is None:
        continue

    if pred.shape != gt.shape:
        gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))

    acc, prec, rec, f1, iou = compute_metrics(gt, pred)

    results.append([pred_file, acc, prec, rec, f1, iou])
    valid_frames += 1


df = pd.DataFrame(results, columns=[
    "Frame", "Accuracy", "Precision", "Recall", "F1-score", "IoU"
])

df.to_csv(frame_csv_path, index=False)

overall_df = pd.DataFrame([{
    "Total Frames": valid_frames,
    "Mean Accuracy": df["Accuracy"].mean() if valid_frames > 0 else 0,
    "Mean Precision": df["Precision"].mean() if valid_frames > 0 else 0,
    "Mean Recall": df["Recall"].mean() if valid_frames > 0 else 0,
    "Mean F1-score": df["F1-score"].mean() if valid_frames > 0 else 0,
    "Mean IoU": df["IoU"].mean() if valid_frames > 0 else 0
}])

overall_df.to_csv(overall_csv_path, index=False)

print("Frames evaluated:", valid_frames)
print("Done!")

Processing Video... Press 'q' to stop.
Done!
Frames evaluated: 4
Done!
